[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 07](README.md)

# Perfilado integral y reproducibilidad

**Tema:** 07 · **Sesiones:** 33, 34 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué fracción del tiempo corresponde a cómputo, comunicación, transferencia, E/S y espera?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** El perfil integral reconcilia cómputo, comunicación, transferencias, sincronización e I/O con el tiempo total y su variabilidad.

**Prerrequisitos.**

- MPI, OpenMP y un modelo de acelerador.
- Afinidad, escalabilidad y lectura de perfiles.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Diseñar una línea temporal de extremo a extremo.
- Resumir repeticiones con mediana y dispersión.
- Relacionar cuellos de botella con evidencia de herramientas.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Un perfil útil comienza con una pregunta y una región delimitada.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Mediana y MAD son robustas frente a algunos outliers, pero los datos crudos se conservan.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Las herramientas cambian el tiempo; la corrida instrumentada explica comportamiento y la corrida ligera cuantifica rendimiento.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- afinidad — vínculo entre trabajo y recursos físicos
- sobresuscripción — más entidades ejecutables que recursos asignados
- perfil — atribución del tiempo a regiones o fases


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.

### Topologia Hibrida

![Nodos con ranks, hilos y GPU local](../../images/topologia-hibrida.svg)

**Cómo leerlo.** Recorre la jerarquía de afuera hacia adentro. El mapeo correcto conserva afinidad local y evita asignar accidentalmente varios ranks al mismo dispositivo.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "07"
NOTEBOOK = "07_hibrido/03_perfilado_reproducible.ipynb"
assert (ROOT / "curso" / "notebooks" / "07_hibrido" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Mediana y MAD

**Situación.** Se resume una muestra sin eliminar observaciones silenciosamente.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
import statistics
samples = [10.2, 10.0, 10.1, 10.3, 10.05, 14.8, 10.15]
median = statistics.median(samples)
mad = statistics.median(abs(value-median) for value in samples)
print({"samples": samples, "median": median, "MAD": mad})
assert mad > 0


### Explicación del resultado

El valor 14.8 se investiga con logs y sistema; no se borra solo porque sea incómodo.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Descomposición del total

**Situación.** Se verifica que las fases y el tiempo no atribuido concuerden con el total.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
phases = {"compute": 48.0, "mpi": 21.0, "h2d_d2h": 12.0, "io": 7.0, "barriers": 6.0}
total = 100.0
unattributed = total - sum(phases.values())
assert unattributed >= 0
for name, percent in phases.items(): print(f"{name:10} {percent:5.1f}%")
print("no atribuido", unattributed, "%")


### Lectura razonada

La suma inferior a 100 % hace visible instrumentación incompleta; una suma superior indica doble conteo.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Cómo distinguirías una fase realmente dominante de una perturbación aislada en una sola repetición?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Definir una pregunta para cada herramienta de perfil.
2. Ejecutar serie ligera y corrida instrumentada separadas.
3. Conservar comando, versiones, trazas y resumen derivado.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Perfilar todo sin región ni hipótesis.
- Comparar tiempos instrumentados con no instrumentados como equivalentes.
- Presentar porcentajes que no reconcilian con el total.


## Criterios de aceptación

- Pregunta y región declaradas.
- Datos crudos trazables al resumen.
- Conclusión respaldada por métrica y no por una captura aislada.


## Síntesis

- La pregunta que debes poder responder es: **¿Qué fracción del tiempo corresponde a cómputo, comunicación, transferencia, E/S y espera?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Protocolo de evidencia](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)
- [Planeación híbrida](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 07](README.md)
